# 1. Importing libraries

In [ ]:
!pip install uv

In [ ]:
!uv pip install datasets transformers huggingface_hub torch tqdm optuna

In [ ]:
import itertools
import random
import string

import torch
from datasets import load_dataset

from tqdm import tqdm

import math
import torch.nn as nn
import torch.nn.functional as F

import optuna                           # <-- ADD THIS

# 2. AI models

In [ ]:
# add code here



# ---- Parallel Scan ----

def parallel_associative_scan(gates, values):
    L = gates.shape[0]
    a = gates.clone()
    b = values.clone()
    stride = 1
    while stride < L:
        idx = torch.arange(stride, L, device=gates.device)
        prev = idx - stride
        new_b = a[idx] * b[prev] + b[idx]
        new_a = a[idx] * a[prev]
        a[idx] = new_a
        b[idx] = new_b
        stride *= 2
    return b

# ---- MambaBlock ----

class MambaBlock(nn.Module):
    def __init__(self, d_model, N, dt_min=0.001, dt_max=0.1, expand=2):
        super().__init__()
        self.d_model = d_model
        self.N = N
        self.dt_min = dt_min
        self.dt_max = dt_max
        self.d_inner = d_model * expand

        A_log_init = torch.tensor([math.log(n + 1) for n in range(N)], dtype=torch.float32)
        self.A_log = nn.Parameter(A_log_init)
        self.b_c = nn.Parameter(torch.zeros(N))

        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.dt_rank = max(1, d_model // 16)
        self.delta_proj    = nn.Linear(self.d_inner, self.dt_rank, bias=False)
        self.delta_proj_up = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        nn.init.constant_(self.delta_proj_up.bias, math.log(math.expm1(1)))

        self.B_proj = nn.Linear(self.d_inner, N, bias=False)
        self.C_proj = nn.Linear(self.d_inner, N, bias=False)

        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=4,
                                groups=self.d_inner, padding=3, bias=True)

    def forward(self, x_seq):
        seq_len = x_seq.shape[0]
        xz = self.in_proj(x_seq)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_branch = x_branch.T.unsqueeze(0)
        x_branch = F.silu(self.conv1d(x_branch)[..., :seq_len])
        x_branch = x_branch.squeeze(0).T

        delta = F.softplus(self.delta_proj_up(self.delta_proj(x_branch))).clamp(self.dt_min, self.dt_max)
        B_all = self.B_proj(x_branch)
        C_all = self.C_proj(x_branch)
        A_diag = -torch.exp(self.A_log)

        A_bar = torch.exp(A_diag.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1))
        B_bar = ((A_bar - 1) / A_diag.unsqueeze(0).unsqueeze(0)) * B_all.unsqueeze(1)
        values = B_bar * x_branch.unsqueeze(-1)

        h_all = parallel_associative_scan(A_bar, values)
        h_all = h_all + self.b_c
        y = (h_all * C_all.unsqueeze(1)).sum(dim=-1)

        output = y * F.silu(z_branch)
        return self.out_proj(output)


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, **kwargs):
        super().__init__()
        self.norm  = RMSNorm(d_model)
        self.mamba = MambaBlock(d_model, N, **kwargs)

    def forward(self, x):
        return x + self.mamba(self.norm(x))


# ---- Mamba encoder backbone (shared by all wrappers) ----

class MambaEncoder(nn.Module):
    """Embedding -> [ResidualBlock x n_layers] -> RMSNorm"""
    def __init__(self, vocab_size, d_model=128, n_layers=2, N=16, expand=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, expand=expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)

    def forward(self, input_ids):
        """input_ids: (B, L) -> (B, L, d_model)"""
        x = self.embedding(input_ids)
        outputs = []
        for b in range(x.shape[0]):
            h = x[b]
            for layer in self.layers:
                h = layer(h)
            h = self.norm_f(h)
            outputs.append(h)
        return torch.stack(outputs, dim=0)


# ---- Wrapper 1: Token-level prediction ----
# For MQAR, Induction Heads, MAD recall tasks, Memorization

class MambaTokenPredictor(nn.Module):
    """Per-position logits over vocab. Use with ignore_index=-100 in CE loss."""
    def __init__(self, vocab_size, d_model=128, n_layers=2, N=16):
        super().__init__()
        self.encoder = MambaEncoder(vocab_size, d_model, n_layers, N)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.encoder(input_ids)          # (B, L, d_model)
        return self.head(h)                  # (B, L, vocab_size)


# ---- Wrapper 2: Sequence-to-fixed-output ----
# For Selective Copying, Compression (input L tokens -> output K tokens)

class MambaSeqToFixed(nn.Module):
    """Reads full sequence, outputs K token predictions from the last K hidden states."""
    def __init__(self, vocab_size, output_len, d_model=128, n_layers=2, N=16):
        super().__init__()
        self.output_len = output_len
        self.encoder = MambaEncoder(vocab_size, d_model, n_layers, N)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.encoder(input_ids)                    # (B, L, d_model)
        h_tail = h[:, -self.output_len:, :]            # (B, K, d_model)
        return self.head(h_tail)                       # (B, K, vocab_size)


# ---- Wrapper 3: Sequence classification ----
# For Parity

class MambaClassifier(nn.Module):
    """Mean-pool -> classify."""
    def __init__(self, vocab_size, num_classes, d_model=128, n_layers=2, N=16):
        super().__init__()
        self.encoder = MambaEncoder(vocab_size, d_model, n_layers, N)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids):
        h = self.encoder(input_ids)          # (B, L, d_model)
        pooled = h.mean(dim=1)               # (B, d_model)
        return self.classifier(pooled)       # (B, num_classes)


# ---- Shared training utility ----

def train_loop(model, train_inputs, train_labels, epochs=20, lr=1e-3, batch_size=32, ignore_index=-100):
    """Simple training loop with progress bars. Returns final loss."""
    device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    n = train_inputs.shape[0]

    for epoch in tqdm(range(1, epochs + 1), desc="Epochs", unit="epoch"):
        model.train()
        perm = torch.randperm(n)
        total_loss = 0
        steps = 0

        batch_iter = tqdm(
            range(0, n, batch_size),
            desc=f"  Epoch {epoch:3d}",
            leave=False,
            unit="batch",
        )
        for i in batch_iter:
            idx = perm[i:i+batch_size]
            x = train_inputs[idx].to(device)
            y = train_labels[idx].to(device)

            logits = model(x)
            logits_flat = logits.reshape(-1, logits.size(-1))
            labels_flat = y.reshape(-1)
            loss = F.cross_entropy(logits_flat, labels_flat, ignore_index=ignore_index)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
            batch_iter.set_postfix(loss=f"{loss.item():.4f}")

        if epoch % 5 == 0 or epoch == 1:
            tqdm.write(f"  Epoch {epoch:3d}  loss = {total_loss/steps:.4f}")

    return total_loss / steps


def eval_accuracy(model, inputs, labels, batch_size=64, ignore_index=-100):
    """Compute accuracy over non-ignored positions."""
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for i in tqdm(range(0, inputs.shape[0], batch_size), desc="Evaluating", leave=False, unit="batch"):
            x = inputs[i:i+batch_size].to(device)
            y = labels[i:i+batch_size].to(device)
            logits = model(x)
            preds = logits.argmax(dim=-1)

            mask = (y != ignore_index)
            correct += (preds[mask] == y[mask]).sum().item()
            total += mask.sum().item()

    acc = correct / total if total > 0 else 0.0
    return acc

# 3. Benchmarks to ensure design works - Create training, validation and test datasets for each benchmark

## 3.1 MQAR — Multi-Query Associative Recall

 - Problem: Prior synthetic recall tests used a single query at a fixed position with tiny vocabularies — unrealistic compared to real language. arXiv MQAR requires multiple key-value lookups per sequence.

 - Why beyond LRA: Directly isolates the recall gap between attention and SSM/convolution models, which explains ~82% of the real-world quality gap.

In [ ]:
def generate_mqar(
    batch_size=64,
    seq_len=128,
    vocab_size=8192,
    num_kv_pairs=4,
    seed=42
):
    """
    Generates MQAR sequences:
    [k1, v1, k2, v2, ..., padding..., q1, q2, ...] -> [v1, v2, ...]
    Keys and queries come from a reserved range; values from the rest.
    """
    random.seed(seed)
    torch.manual_seed(seed)

    key_vocab_start = vocab_size  # keys live above normal vocab
    inputs = torch.zeros(batch_size, seq_len, dtype=torch.long)
    labels = torch.full((batch_size, seq_len), -100, dtype=torch.long)  # -100 = ignore

    for b in range(batch_size):
        # Generate unique key-value pairs
        keys = random.sample(range(key_vocab_start, key_vocab_start + 256), num_kv_pairs)
        values = random.sample(range(1, vocab_size), num_kv_pairs)

        # Place KV pairs at the start
        for i, (k, v) in enumerate(zip(keys, values)):
            inputs[b, 2 * i] = k
            inputs[b, 2 * i + 1] = v

        # Fill middle with random padding tokens
        kv_end = 2 * num_kv_pairs
        query_start = seq_len - num_kv_pairs
        inputs[b, kv_end:query_start] = torch.randint(1, vocab_size, (query_start - kv_end,))

        # Place queries at the end, labels are the expected values
        for i, (k, v) in enumerate(zip(keys, values)):
            inputs[b, query_start + i] = k
            labels[b, query_start + i] = v

    return inputs, labels

# Generate a batch
inputs, labels = generate_mqar(batch_size=32, seq_len=64, num_kv_pairs=4)
print(f"Inputs shape:  {inputs.shape}")
print(f"Labels shape:  {labels.shape}")
print(f"Sample input:  {inputs[0, :12]}")  # first 4 KV pairs + some padding
print(f"Sample labels: {labels[0, -4:]}")  # expected answers for 4 queries

In [ ]:
train_inp, train_lab = generate_mqar(batch_size=512, seq_len=128, num_kv_pairs=4, seed=42)
val_inp,   val_lab   = generate_mqar(batch_size=128, seq_len=128, num_kv_pairs=4, seed=99)
test_inp,  test_lab  = generate_mqar(batch_size=128, seq_len=128, num_kv_pairs=4, seed=7)

MQAR_VOCAB = 8192 + 256

mqar_search_space = {
    "d_model":  [64, 128],
    "n_layers": [2, 4],
    "N":        [8, 16],
    "lr":       [1e-3, 5e-4],
}

def objective_mqar(trial):
    d_model  = trial.suggest_categorical("d_model",  mqar_search_space["d_model"])
    n_layers = trial.suggest_categorical("n_layers", mqar_search_space["n_layers"])
    N        = trial.suggest_categorical("N",        mqar_search_space["N"])
    lr       = trial.suggest_categorical("lr",       mqar_search_space["lr"])

    model = MambaTokenPredictor(vocab_size=MQAR_VOCAB, d_model=d_model, n_layers=n_layers, N=N)
    train_loop(model, train_inp, train_lab, epochs=10, lr=lr, batch_size=32)
    return eval_accuracy(model, val_inp, val_lab)

study_mqar = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(mqar_search_space),
    study_name="mqar-grid",
)
study_mqar.optimize(objective_mqar, n_trials=2)

best = study_mqar.best_trial.params
print(f"MQAR best val_acc: {study_mqar.best_trial.value:.4f} | Params: {best}")

model_mqar = MambaTokenPredictor(vocab_size=MQAR_VOCAB, d_model=best["d_model"], n_layers=best["n_layers"], N=best["N"])
print("=== Training best Mamba on MQAR ===")
train_loop(model_mqar, train_inp, train_lab, epochs=20, lr=best["lr"])
acc = eval_accuracy(model_mqar, test_inp, test_lab)
print(f"MQAR test accuracy: {acc:.4f}")

## 3.2 Selective Copying

 - Problem: Standard copying tasks could be solved with fixed convolution kernels using positional tricks. HackerNoon Randomizing token positions forces content-aware selection.

 - Why beyond LRA: Tests input-dependent gating — the core innovation in Mamba/S6 — which LRA doesn't isolate.

In [ ]:
def generate_selective_copying(
    batch_size=64,
    seq_len=128,
    num_tokens_to_copy=8,
    vocab_size=16,
    blank_token=0,
    marker_token=None,
    seed=42
):
    """
    Selective Copying: a sequence has `num_tokens_to_copy` meaningful tokens
    scattered at random positions among blank tokens. The model must output
    only the meaningful tokens in order.

    Unlike fixed-interval copying, positions are random — so the model must
    use content-aware (selective) reasoning, not just fixed offsets.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if marker_token is None:
        marker_token = vocab_size  # special token outside vocab

    inputs = torch.full((batch_size, seq_len), blank_token, dtype=torch.long)
    labels = torch.zeros(batch_size, num_tokens_to_copy, dtype=torch.long)

    for b in range(batch_size):
        # Pick random positions for the meaningful tokens
        positions = sorted(random.sample(range(seq_len - 1), num_tokens_to_copy))
        tokens = torch.randint(1, vocab_size, (num_tokens_to_copy,))

        for i, pos in enumerate(positions):
            inputs[b, pos] = tokens[i]
            labels[b, i] = tokens[i]

        # Append a marker token at the end to signal "now reproduce"
        inputs[b, -1] = marker_token

    return inputs, labels

inputs, labels = generate_selective_copying(batch_size=32, seq_len=64, num_tokens_to_copy=6)
print("=== Selective Copying ===")
print(f"Inputs shape:  {inputs.shape}")
print(f"Labels shape:  {labels.shape}")
print(f"Sample input:  {inputs[0]}")
print(f"Expected copy: {labels[0]}")

In [ ]:
# add code here

NUM_COPY = 6

train_inp, train_lab = generate_selective_copying(batch_size=512, seq_len=128, num_tokens_to_copy=NUM_COPY, seed=42)
val_inp,   val_lab   = generate_selective_copying(batch_size=128, seq_len=128, num_tokens_to_copy=NUM_COPY, seed=99)
test_inp,  test_lab  = generate_selective_copying(batch_size=128, seq_len=128, num_tokens_to_copy=NUM_COPY, seed=7)

selcopy_search_space = {
    "d_model":  [64, 128],
    "n_layers": [2, 4],
    "N":        [8, 16],
    "lr":       [1e-3, 5e-4],
}

def objective_selcopy(trial):
    d_model  = trial.suggest_categorical("d_model",  selcopy_search_space["d_model"])
    n_layers = trial.suggest_categorical("n_layers", selcopy_search_space["n_layers"])
    N        = trial.suggest_categorical("N",        selcopy_search_space["N"])
    lr       = trial.suggest_categorical("lr",       selcopy_search_space["lr"])

    model = MambaSeqToFixed(vocab_size=17, output_len=NUM_COPY, d_model=d_model, n_layers=n_layers, N=N)
    train_loop(model, train_inp, train_lab, epochs=10, lr=lr, batch_size=32, ignore_index=-1)
    return eval_accuracy(model, val_inp, val_lab, ignore_index=-1)

study_selcopy = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(selcopy_search_space),
    study_name="selcopy-grid",
)
study_selcopy.optimize(objective_selcopy, n_trials=2)

best = study_selcopy.best_trial.params
print(f"Selective Copying best val_acc: {study_selcopy.best_trial.value:.4f} | Params: {best}")

model_selcopy = MambaSeqToFixed(vocab_size=17, output_len=NUM_COPY, d_model=best["d_model"], n_layers=best["n_layers"], N=best["N"])
print("=== Training best Mamba on Selective Copying ===")
train_loop(model_selcopy, train_inp, train_lab, epochs=20, lr=best["lr"], ignore_index=-1)
acc = eval_accuracy(model_selcopy, test_inp, test_lab, ignore_index=-1)
print(f"Selective Copying test accuracy: {acc:.4f}")

## 3.3 Induction Heads

 - Problem: Needed to understand the mechanism behind in-context learning. arXiv Found that induction heads ([A][B]...[A]→[B]) are likely the primary circuit driving ICL.

 - Why beyond LRA: Probes the specific two-layer copy-and-recall circuit that underpins in-context learning — critical for evaluating SSMs vs. transformers.

In [ ]:
def generate_induction_heads(
    batch_size=64,
    seq_len=128,
    vocab_size=64,
    num_triggers=4,
    seed=42
):
    """
    Induction Head task: earlier in the sequence, pattern "A B" appears.
    Later, "A" appears again — the model must predict "B".

    This tests the two-layer attention circuit:
      Layer 1 attends to the previous token of the current query.
      Layer 2 copies the token that followed the previous occurrence.
    """
    random.seed(seed)
    torch.manual_seed(seed)

    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = torch.full((batch_size, seq_len), -100, dtype=torch.long)  # -100 = ignore

    for b in range(batch_size):
        # Create trigger pairs (A, B) in the first half
        pair_positions = sorted(random.sample(range(0, seq_len // 2 - 1), num_triggers))
        pairs = []
        for pos in pair_positions:
            a = random.randint(1, vocab_size - 1)
            btoken = random.randint(1, vocab_size - 1)
            inputs[b, pos] = a
            inputs[b, pos + 1] = btoken
            pairs.append((a, btoken))

        # Place query "A" tokens in the second half; label is "B"
        query_positions = sorted(random.sample(
            range(seq_len // 2, seq_len - 1), num_triggers
        ))
        for i, qpos in enumerate(query_positions):
            a, btoken = pairs[i]
            inputs[b, qpos] = a
            labels[b, qpos] = btoken  # model should predict B here

    return inputs, labels

inputs, labels = generate_induction_heads(batch_size=32, seq_len=64, num_triggers=3)
print("=== Induction Heads ===")
print(f"Inputs shape:  {inputs.shape}")
print(f"Labels shape:  {labels.shape}")

# Show the trigger positions (where label != -100)
sample_idx = 0
trigger_mask = labels[sample_idx] != -100
trigger_pos = trigger_mask.nonzero().squeeze()
print(f"Query positions: {trigger_pos.tolist()}")
print(f"Expected tokens: {labels[sample_idx, trigger_mask].tolist()}")

In [ ]:
# add code here

train_inp, train_lab = generate_induction_heads(batch_size=512, seq_len=128, num_triggers=4, seed=42)
val_inp,   val_lab   = generate_induction_heads(batch_size=128, seq_len=128, num_triggers=4, seed=99)
test_inp,  test_lab  = generate_induction_heads(batch_size=128, seq_len=128, num_triggers=4, seed=7)

induction_search_space = {
    "d_model":  [64, 128],
    "n_layers": [2, 4],
    "N":        [8, 16],
    "lr":       [1e-3, 5e-4],
}

def objective_induction(trial):
    d_model  = trial.suggest_categorical("d_model",  induction_search_space["d_model"])
    n_layers = trial.suggest_categorical("n_layers", induction_search_space["n_layers"])
    N        = trial.suggest_categorical("N",        induction_search_space["N"])
    lr       = trial.suggest_categorical("lr",       induction_search_space["lr"])

    model = MambaTokenPredictor(vocab_size=64, d_model=d_model, n_layers=n_layers, N=N)
    train_loop(model, train_inp, train_lab, epochs=10, lr=lr, batch_size=32)
    return eval_accuracy(model, val_inp, val_lab)

study_induction = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(induction_search_space),
    study_name="induction-grid",
)
study_induction.optimize(objective_induction, n_trials=2)

best = study_induction.best_trial.params
print(f"Induction best val_acc: {study_induction.best_trial.value:.4f} | Params: {best}")

model_induction = MambaTokenPredictor(vocab_size=64, d_model=best["d_model"], n_layers=best["n_layers"], N=best["N"])
print("=== Training best Mamba on Induction Heads ===")
train_loop(model_induction, train_inp, train_lab, epochs=20, lr=best["lr"])
acc = eval_accuracy(model_induction, test_inp, test_lab)
print(f"Induction Heads test accuracy: {acc:.4f}")

## 3.4 MAD Benchmark (6 sub-tasks)

 - Problem: Architecture design was too expensive; needed cheap proxy tasks predictive of scaling laws. arXiv

 - Why beyond LRA: Decomposes capabilities into orthogonal skills. MAD scores correlate with compute-optimal perplexity arXiv, enabling fast architecture prototyping without full-scale training.

In [ ]:
def mad_in_context_recall(batch_size=32, seq_len=128, vocab_size=64, num_kv=8, seed=42):
    """Exact key-value recall from context."""
    random.seed(seed); torch.manual_seed(seed)
    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = torch.full((batch_size, seq_len), -100)
    for b in range(batch_size):
        keys = random.sample(range(vocab_size, vocab_size + 128), num_kv)
        vals = [random.randint(1, vocab_size - 1) for _ in range(num_kv)]
        for i, (k, v) in enumerate(zip(keys, vals)):
            inputs[b, 2*i] = k
            inputs[b, 2*i+1] = v
        for i, (k, v) in enumerate(zip(keys, vals)):
            qpos = seq_len - num_kv + i
            inputs[b, qpos] = k
            labels[b, qpos] = v
    return inputs, labels

In [ ]:
def mad_fuzzy_recall(batch_size=32, seq_len=128, vocab_size=64, num_kv=8, noise_frac=0.3, seed=42):
    """Recall with noisy/corrupted keys at query time."""
    random.seed(seed); torch.manual_seed(seed)
    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = torch.full((batch_size, seq_len), -100)
    for b in range(batch_size):
        keys = [random.randint(vocab_size, vocab_size + 127) for _ in range(num_kv)]
        vals = [random.randint(1, vocab_size - 1) for _ in range(num_kv)]
        for i, (k, v) in enumerate(zip(keys, vals)):
            inputs[b, 2*i] = k
            inputs[b, 2*i+1] = v
        for i, (k, v) in enumerate(zip(keys, vals)):
            qpos = seq_len - num_kv + i
            # Corrupt the key slightly
            noisy_key = k + (random.randint(-3, 3) if random.random() < noise_frac else 0)
            inputs[b, qpos] = max(1, noisy_key)
            labels[b, qpos] = v
    return inputs, labels

In [ ]:
def mad_noisy_recall(batch_size=32, seq_len=128, vocab_size=64, num_kv=8, seed=42):
    """Recall with extra distractor KV pairs injected."""
    random.seed(seed); torch.manual_seed(seed)
    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = torch.full((batch_size, seq_len), -100)
    for b in range(batch_size):
        keys = random.sample(range(vocab_size, vocab_size + 128), num_kv)
        vals = [random.randint(1, vocab_size - 1) for _ in range(num_kv)]
        # Place real pairs
        for i, (k, v) in enumerate(zip(keys, vals)):
            inputs[b, 2*i] = k
            inputs[b, 2*i+1] = v
        # Inject distractors in the middle
        mid = 2 * num_kv
        for j in range(mid, mid + num_kv):
            inputs[b, j] = random.randint(vocab_size + 128, vocab_size + 255)
            inputs[b, j] = random.randint(1, vocab_size - 1)
        # Queries
        for i, (k, v) in enumerate(zip(keys, vals)):
            qpos = seq_len - num_kv + i
            inputs[b, qpos] = k
            labels[b, qpos] = v
    return inputs, labels

In [ ]:
def mad_selective_copying(batch_size=32, seq_len=128, vocab_size=16, num_copy=8, seed=42):
    """Copy only marked tokens from random positions."""
    random.seed(seed); torch.manual_seed(seed)
    blank = 0; marker = vocab_size
    inputs = torch.full((batch_size, seq_len), blank)
    labels = torch.zeros(batch_size, num_copy, dtype=torch.long)
    for b in range(batch_size):
        positions = sorted(random.sample(range(seq_len - 1), num_copy))
        tokens = torch.randint(1, vocab_size, (num_copy,))
        for i, pos in enumerate(positions):
            inputs[b, pos] = tokens[i]
            labels[b, i] = tokens[i]
        inputs[b, -1] = marker
    return inputs, labels

In [ ]:
def mad_compression(batch_size=32, seq_len=128, vocab_size=32, pattern_len=8, repeats=4, seed=42):
    """Detect and compress a repeated pattern in the sequence."""
    random.seed(seed); torch.manual_seed(seed)
    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = torch.zeros(batch_size, pattern_len, dtype=torch.long)
    for b in range(batch_size):
        pattern = torch.randint(1, vocab_size, (pattern_len,))
        for r in range(repeats):
            start = r * pattern_len
            if start + pattern_len <= seq_len:
                inputs[b, start:start+pattern_len] = pattern
        labels[b] = pattern  # model should output the underlying pattern
    return inputs, labels

In [ ]:
def mad_memorization(batch_size=32, seq_len=128, vocab_size=64, seed=42):
    """Memorize and reproduce the entire input sequence."""
    torch.manual_seed(seed)
    inputs = torch.randint(1, vocab_size, (batch_size, seq_len))
    labels = inputs.clone()
    return inputs, labels

In [ ]:
tasks = {
    "In-Context Recall": mad_in_context_recall(),
    "Fuzzy Recall":      mad_fuzzy_recall(),
    "Noisy Recall":      mad_noisy_recall(),
    "Selective Copying":  mad_selective_copying(),
    "Compression":       mad_compression(),
    "Memorization":      mad_memorization(),
}

In [ ]:
for name, (inp, lab) in tasks.items():
    print(f"{name:20s} | inputs: {inp.shape}  labels: {lab.shape}")

In [ ]:
# add code here

mad_train = {
    "In-Context Recall": mad_in_context_recall(batch_size=512, seed=42),
    "Fuzzy Recall":      mad_fuzzy_recall(batch_size=512, seed=42),
    "Noisy Recall":      mad_noisy_recall(batch_size=512, seed=42),
    "Selective Copying":  mad_selective_copying(batch_size=512, seed=42),
    "Compression":       mad_compression(batch_size=512, seed=42),
    "Memorization":      mad_memorization(batch_size=512, seed=42),
}
mad_val = {
    "In-Context Recall": mad_in_context_recall(batch_size=128, seed=99),
    "Fuzzy Recall":      mad_fuzzy_recall(batch_size=128, seed=99),
    "Noisy Recall":      mad_noisy_recall(batch_size=128, seed=99),
    "Selective Copying":  mad_selective_copying(batch_size=128, seed=99),
    "Compression":       mad_compression(batch_size=128, seed=99),
    "Memorization":      mad_memorization(batch_size=128, seed=99),
}
mad_test = {
    "In-Context Recall": mad_in_context_recall(batch_size=128, seed=7),
    "Fuzzy Recall":      mad_fuzzy_recall(batch_size=128, seed=7),
    "Noisy Recall":      mad_noisy_recall(batch_size=128, seed=7),
    "Selective Copying":  mad_selective_copying(batch_size=128, seed=7),
    "Compression":       mad_compression(batch_size=128, seed=7),
    "Memorization":      mad_memorization(batch_size=128, seed=7),
}

token_tasks = ["In-Context Recall", "Fuzzy Recall", "Noisy Recall", "Memorization"]
fixed_tasks = {"Selective Copying": 8, "Compression": 8}

mad_search_space = {
    "d_model":  [64, 128],
    "n_layers": [2, 4],
    "N":        [8, 16],
    "lr":       [1e-3, 5e-4],
}

mad_results = {}
mad_studies = {}

for name in token_tasks:
    print(f"\n=== MAD Grid Search: {name} ===")
    tr_inp, tr_lab = mad_train[name]
    va_inp, va_lab = mad_val[name]
    te_inp, te_lab = mad_test[name]
    vocab = 64 + 128

    def make_objective_token(tr_i, tr_l, va_i, va_l, v):
        def objective(trial):
            d_model  = trial.suggest_categorical("d_model",  mad_search_space["d_model"])
            n_layers = trial.suggest_categorical("n_layers", mad_search_space["n_layers"])
            N        = trial.suggest_categorical("N",        mad_search_space["N"])
            lr       = trial.suggest_categorical("lr",       mad_search_space["lr"])
            model = MambaTokenPredictor(vocab_size=v, d_model=d_model, n_layers=n_layers, N=N)
            train_loop(model, tr_i, tr_l, epochs=10, lr=lr, batch_size=32)
            return eval_accuracy(model, va_i, va_l)
        return objective

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.GridSampler(mad_search_space),
        study_name=f"mad-{name.lower().replace(' ','-')}-grid",
    )
    study.optimize(make_objective_token(tr_inp, tr_lab, va_inp, va_lab, vocab), n_trials=2)

    best = study.best_trial.params
    print(f"  {name} best val_acc: {study.best_trial.value:.4f} | {best}")

    model = MambaTokenPredictor(vocab_size=vocab, d_model=best["d_model"], n_layers=best["n_layers"], N=best["N"])
    train_loop(model, tr_inp, tr_lab, epochs=20, lr=best["lr"])
    acc = eval_accuracy(model, te_inp, te_lab)
    mad_results[name] = acc
    mad_studies[name] = study
    print(f"  {name} test accuracy: {acc:.4f}")

for name, out_len in fixed_tasks.items():
    print(f"\n=== MAD Grid Search: {name} ===")
    tr_inp, tr_lab = mad_train[name]
    va_inp, va_lab = mad_val[name]
    te_inp, te_lab = mad_test[name]
    vocab = 17 if name == "Selective Copying" else 32

    def make_objective_fixed(tr_i, tr_l, va_i, va_l, v, ol):
        def objective(trial):
            d_model  = trial.suggest_categorical("d_model",  mad_search_space["d_model"])
            n_layers = trial.suggest_categorical("n_layers", mad_search_space["n_layers"])
            N        = trial.suggest_categorical("N",        mad_search_space["N"])
            lr       = trial.suggest_categorical("lr",       mad_search_space["lr"])
            model = MambaSeqToFixed(vocab_size=v, output_len=ol, d_model=d_model, n_layers=n_layers, N=N)
            train_loop(model, tr_i, tr_l, epochs=10, lr=lr, batch_size=32, ignore_index=-1)
            return eval_accuracy(model, va_i, va_l, ignore_index=-1)
        return objective

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.GridSampler(mad_search_space),
        study_name=f"mad-{name.lower().replace(' ','-')}-grid",
    )
    study.optimize(make_objective_fixed(tr_inp, tr_lab, va_inp, va_lab, vocab, out_len), n_trials=2)

    best = study.best_trial.params
    print(f"  {name} best val_acc: {study.best_trial.value:.4f} | {best}")

    model = MambaSeqToFixed(vocab_size=vocab, output_len=out_len, d_model=best["d_model"], n_layers=best["n_layers"], N=best["N"])
    train_loop(model, tr_inp, tr_lab, epochs=20, lr=best["lr"], ignore_index=-1)
    acc = eval_accuracy(model, te_inp, te_lab, ignore_index=-1)
    mad_results[name] = acc
    mad_studies[name] = study
    print(f"  {name} test accuracy: {acc:.4f}")

print("\n=== MAD Summary ===")
for name, acc in mad_results.items():
    print(f"  {name:20s}  {acc:.4f}  best_params: {mad_studies[name].best_trial.params}")

## 3.5 State Tracking — Parity / Permutation

Problem: Even the simplest state-tracking task — computing parity of a bit sequence — cannot be solved by current linear RNNs arXiv, exposing fundamental expressivity limits of SSMs and transformers on sequential computation.

Why beyond LRA: Tests whether models can maintain and update internal state over time — a prerequisite for code evaluation, entity tracking, and logical reasoning that LRA doesn't cover.

In [ ]:
def generate_parity_data(n_samples=1000, seq_len=50):
    data = []
    for _ in range(n_samples):
        bits = [random.randint(0, 1) for _ in range(seq_len)]
        label = sum(bits) % 2
        data.append((bits, label))
    return data

parity_data = generate_parity_data()
print(f"Parity samples: {len(parity_data)}, first: {parity_data[0]}")


In [ ]:
# add code here

train_parity = generate_parity_data(n_samples=2000, seq_len=50)
test_parity  = generate_parity_data(n_samples=500, seq_len=50)
val_parity   = generate_parity_data(n_samples=500, seq_len=50)

train_inp = torch.tensor([bits for bits, _ in train_parity], dtype=torch.long)
train_lab = torch.tensor([lab for _, lab in train_parity], dtype=torch.long)
test_inp  = torch.tensor([bits for bits, _ in test_parity], dtype=torch.long)
test_lab  = torch.tensor([lab for _, lab in test_parity], dtype=torch.long)
val_inp   = torch.tensor([bits for bits, _ in val_parity], dtype=torch.long)
val_lab   = torch.tensor([lab for _, lab in val_parity], dtype=torch.long)

parity_search_space = {
    "d_model":  [64, 128],
    "n_layers": [2, 4],
    "N":        [8, 16],
    "lr":       [1e-3, 5e-4],
}

def objective_parity(trial):
    d_model  = trial.suggest_categorical("d_model",  parity_search_space["d_model"])
    n_layers = trial.suggest_categorical("n_layers", parity_search_space["n_layers"])
    N        = trial.suggest_categorical("N",        parity_search_space["N"])
    lr       = trial.suggest_categorical("lr",       parity_search_space["lr"])

    model = MambaClassifier(vocab_size=2, num_classes=2, d_model=d_model, n_layers=n_layers, N=N)
    train_loop(model, train_inp, train_lab, epochs=15, lr=lr, batch_size=32, ignore_index=-1)

    model.eval()
    with torch.no_grad():
        logits = model(val_inp)
        preds = logits.argmax(dim=-1)
        acc = (preds == val_lab).float().mean().item()
    return acc

study_parity = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(parity_search_space),
    study_name="parity-grid",
)
study_parity.optimize(objective_parity, n_trials=2)

best = study_parity.best_trial.params
print(f"Parity best val_acc: {study_parity.best_trial.value:.4f} | Params: {best}")

model_parity = MambaClassifier(vocab_size=2, num_classes=2, d_model=best["d_model"], n_layers=best["n_layers"], N=best["N"])
print("=== Training best Mamba on Parity ===")
train_loop(model_parity, train_inp, train_lab, epochs=30, lr=best["lr"], batch_size=32, ignore_index=-1)

model_parity.eval()
with torch.no_grad():
    logits = model_parity(test_inp)
    preds = logits.argmax(dim=-1)
    acc = (preds == test_lab).float().mean().item()

print(f"Parity test accuracy: {acc:.4f}")

# X. Complete results

In [ ]:
"""## 4. Full Results Summary"""

print("\n" + "=" * 70)
print("  GRID SEARCH RESULTS SUMMARY (2 trials each)")
print("=" * 70)

for name, study_obj in [
    ("MQAR",              study_mqar),
    ("Selective Copying",  study_selcopy),
    ("Induction Heads",   study_induction),
    ("Parity",            study_parity),
]:
    print(f"\n  {name:20s}  best val_acc = {study_obj.best_trial.value:.4f}")
    print(f"  {'':20s}  params: {study_obj.best_trial.params}")

print(f"\n  --- MAD sub-tasks ---")
for name, study_obj in mad_studies.items():
    print(f"  {name:20s}  best val_acc = {study_obj.best_trial.value:.4f}")
    print(f"  {'':20s}  params: {study_obj.best_trial.params}")

print("\n" + "=" * 70)